In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [0]:
# Load Features
train_df = pd.read_csv("train_trip_duration_features.csv")
test_df = pd.read_csv("test_trip_duration_features.csv")

TARGET = 'trip_duration_minutes'

X_train = train_df.drop(columns = [TARGET])
y_train = train_df[TARGET]

X_test = test_df.drop(columns = [TARGET])
y_test = test_df[TARGET]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")



In [0]:
# Define column types
categorical_cols = ['priority_level', 'status','payment_type','booking_source']

numeric_cols = ['fare_amount','distance','expected_vs_actual_pickup_minutes_difference','dispatch_to_arrival_minutes','month','quarter', 'year', 'weekend_flag',  'pickup_latitude', 'pickup_longitude', 'fare_per_distance', 'pickup_delay_signal', 'journey_urgency', 'lat_delta', 'lon_delta', 'location_distance_proxy']



In [0]:

# sklearn preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
)

# Model pipeline
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())   
])

In [0]:
# MLflow experiement and train inside tracked run
mlflow.set_experiment("/team1-trip-duration-prediction")

# Handle NaN values (journey_urgency has missing values)
X_train = X_train.fillna(X_train.median(numeric_only=True))
X_test = X_test.fillna(X_test.median(numeric_only=True))

with mlflow.start_run(run_name="linear_regression_baseline") as run:
    # Log parameters
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("num_features", len(numeric_cols) + len(categorical_cols))
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("target", TARGET)

    # Train
    lr_pipeline.fit(X_train, y_train)

    # Predict
    y_pred = lr_pipeline.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    # Log metrics
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    # Log the full pipeline (preprocessor + model)
    mlflow.sklearn.log_model(lr_pipeline, "model")

    run_id = run.info.run_id
    print(f"Run ID: {run_id}")
    print(f"MAE:  {mae:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R²:   {r2:.3f}")

In [0]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.3, s=10, color='purple')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--', color='hotpink', label='Perfect')
plt.xlabel("Actual Trip Duration (min)")
plt.ylabel("Predicted Trip Duration (min)")
plt.title("Linear Regression: Predicted vs Actual")
plt.legend()
plt.show()

In [0]:
from mlflow.models import infer_signature

model_name = "students_data.team1_taxi.trip_duration_model"

# Re-log model with signature (required by Unity Catalog)
signature = infer_signature(X_test, y_pred)

with mlflow.start_run(run_id=run_id):
    mlflow.sklearn.log_model(lr_pipeline, "model", signature=signature)

model_uri = f"runs:/{run_id}/model"
result = mlflow.register_model(model_uri, model_name)
print(f"Model registered: {model_name}, version {result.version}")